# 4.5.1 Un méta-modèle appris plutôt qu'un arbitre : Dataset 2 (extrait final, 60 000 images)

Un arbitre top1/top2 ne peut rien si la bonne classe n'est ni 1re ni 2e. On essaie donc un
méta-modèle (régression logistique) sur les documents incertains : il voit les 6 scores du SVM,
l'écart entre les deux meilleurs et 10 mesures d'image, et ne remplace le SVM que s'il est très sûr
de lui. Les scores du SVM viennent de 4.2 (`scores_svm_60k.csv`).

In [1]:
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.contingency_tables import mcnemar

from outils_4 import ARTEFACTS_60K, CLASSES, table_features, top_two

SCORES = [f"score_{c}" for c in CLASSES]
scores = pd.read_csv(f"{ARTEFACTS_60K}/scores_svm_60k.csv")
scores = pd.concat([scores, top_two(scores[SCORES].to_numpy(), CLASSES)], axis=1)

## 1. Le plafond d'un arbitre top1/top2

In [2]:
test = scores[scores["ensemble"] == "test"]
seuil = np.sort(test["ecart"])[799] + 1e-9  # les 800 documents de test les plus incertains
zone = test[test["ecart"] < seuil]
dans_top2 = (zone["top1"] == zone["label"]) | (zone["top2"] == zone["label"])
print(f"Zone incertaine du test : {len(zone)} documents, SVM juste sur {(zone['top1'] == zone['label']).sum()}")
print(f"Arbitre top1/top2 parfait : au mieux {dans_top2.sum()}/{len(zone)} = {dans_top2.mean():.2%}")

Zone incertaine du test : 800 documents, SVM juste sur 510
Arbitre top1/top2 parfait : au mieux 682/800 = 85.25%


## 2. Les documents incertains, découpés en 3

Entraînement (60 %), validation (20 %, pour le seuil de confiance) et test (20 %, une seule mesure).
Les 10 mesures d'image sont calculées une fois (2 minutes) puis enregistrées.

In [3]:
incertains = scores[scores["ecart"] < seuil].reset_index(drop=True)

fichier_mesures = f"{ARTEFACTS_60K}/meta_features.csv"
if not os.path.exists(fichier_mesures):
    mesures = pd.DataFrame([table_features("../" + f) for f in incertains["filepath"]], index=incertains["filepath"])
    boites = pd.read_parquet(f"{ARTEFACTS_60K}/ocr_boxes.parquet")
    page = boites.groupby("doc_id")[["img_w", "img_h"]].first()
    mots = boites[(boites["text"].str.strip() != "") & (boites["conf"] >= 60)]
    g = mots.groupby("doc_id")
    mesures["aspect_ratio"] = page["img_h"] / page["img_w"]
    mesures["text_density"] = (mots["width"] * mots["height"]).groupby(mots["doc_id"]).sum() / (page["img_w"] * page["img_h"])
    mesures["n_blocks"] = g["block_num"].nunique()
    mesures["n_lines"] = mots.drop_duplicates(["doc_id", "block_num", "par_num", "line_num"]).groupby("doc_id").size()
    mesures["first_block_y"] = g["top"].min() / page["img_h"]  # où commence le texte (0 = tout en haut)
    mesures["median_font_size"] = g["height"].median()
    mesures = mesures.fillna({"text_density": 0, "n_blocks": 0, "n_lines": 0, "first_block_y": 1, "median_font_size": 0})
    mesures.reset_index().to_csv(fichier_mesures, index=False)

incertains = incertains.merge(pd.read_csv(fichier_mesures), on="filepath", how="left")
MESURES = ["ratio_lignes_h", "ratio_lignes_v", "n_lignes_h", "n_lignes_v", "aspect_ratio",
           "text_density", "n_blocks", "n_lines", "first_block_y", "median_font_size"]
VARIABLES = SCORES + ["ecart"] + MESURES

meta_train, reste = train_test_split(incertains, test_size=0.4, random_state=42, stratify=incertains["label"])
meta_val, meta_test = train_test_split(reste, test_size=0.5, random_state=42, stratify=reste["label"])
print(f"Documents incertains : {len(incertains)} -> entraînement {len(meta_train)}, validation {len(meta_val)}, test {len(meta_test)}")

Documents incertains : 4239 -> entraînement 2543, validation 848, test 848


## 3. Le méta-modèle

In [4]:
meta = LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42)
meta.fit(meta_train[VARIABLES], meta_train["label"])

print(f"Validation, SVM seul : {(meta_val['top1'] == meta_val['label']).mean():.1%} | "
      f"méta-modèle seul : {(meta.predict(meta_val[VARIABLES]) == meta_val['label']).mean():.1%}")

Validation, SVM seul : 66.2% | méta-modèle seul : 66.0%


Seul, il ne fait pas mieux que le SVM. On ne le suit donc que s'il dépasse un seuil de confiance
`tau`, choisi sur la validation.

In [5]:
def replace_if_confident(donnees, tau, modele):
    """on suit le méta-modèle s'il est sûr de lui (proba > tau), sinon le SVM"""
    probas = modele.predict_proba(donnees[VARIABLES])
    return np.where(probas.max(axis=1) > tau, modele.classes_[probas.argmax(axis=1)], donnees["top1"])


def count_changes(donnees, prediction):
    """combien de documents corrigés et cassés par rapport au SVM seul"""
    svm_juste, juste = donnees["top1"] == donnees["label"], prediction == donnees["label"]
    return (~svm_juste & juste).sum(), (svm_juste & ~juste).sum()


lignes = []
for tau in np.round(np.arange(0.5, 0.96, 0.05), 2):
    corriges, casses = count_changes(meta_val, replace_if_confident(meta_val, tau, meta))
    lignes.append({"tau": tau, "corrigés": corriges, "cassés": casses, "gain net": corriges - casses})
balayage = pd.DataFrame(lignes)
tau = balayage.loc[balayage["gain net"].idxmax(), "tau"]
print(f"Seuil retenu sur la validation : tau = {tau}")
balayage

Seuil retenu sur la validation : tau = 0.8


,tau,corrigés,cassés,gain net
0,0.50,22,27,-5
1,0.55,17,21,-4
2,0.60,14,17,-3
3,0.65,12,13,-1
4,0.70,9,9,0
5,0.75,7,6,1
6,0.80,5,2,3
7,0.85,2,2,0
8,0.90,1,1,0
9,0.95,1,1,0


## 4. Une seule mesure sur le test, et un test statistique

McNemar compare deux modèles sur les documents où ils ne sont pas d'accord (corrigés contre cassés).
Si p > 0,05, l'écart peut venir du hasard.

In [6]:
prediction = replace_if_confident(meta_test, tau, meta)
corriges, casses = count_changes(meta_test, prediction)
svm_juste = (meta_test["top1"] == meta_test["label"]).to_numpy()
juste = (prediction == meta_test["label"]).to_numpy()
print(f"Test ({len(meta_test)} documents) : SVM seul {svm_juste.sum()} justes -> avec le méta-modèle {juste.sum()}")
print(f"Corrigés : {corriges} | cassés : {casses} | gain net : {corriges - casses:+d}")

tableau = [[(svm_juste & juste).sum(), (svm_juste & ~juste).sum()],
           [(~svm_juste & juste).sum(), (~svm_juste & ~juste).sum()]]
print(f"Test de McNemar : p = {mcnemar(tableau, exact=True).pvalue:.4f}")

Test (848 documents) : SVM seul 538 justes -> avec le méta-modèle 546
Corrigés : 8 | cassés : 0 | gain net : +8
Test de McNemar : p = 0.0078


## 5. Limite : un gain fragile

La même chose, avec les variables standardisées avant la régression logistique.

In [7]:
meta_std = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42))
meta_std.fit(meta_train[VARIABLES], meta_train["label"])
gains = {t: np.subtract(*count_changes(meta_val, replace_if_confident(meta_val, t, meta_std)))
         for t in np.round(np.arange(0.5, 0.96, 0.05), 2)}
tau_std = max(gains, key=gains.get)

prediction = replace_if_confident(meta_test, tau_std, meta_std)
corriges, casses = count_changes(meta_test, prediction)
juste = (prediction == meta_test["label"]).to_numpy()
tableau = [[(svm_juste & juste).sum(), (svm_juste & ~juste).sum()], [(~svm_juste & juste).sum(), (~svm_juste & ~juste).sum()]]
print(f"Variables standardisées (tau = {tau_std}) : corrigés {corriges}, cassés {casses}, "
      f"gain net {corriges - casses:+d}, McNemar p = {mcnemar(tableau, exact=True).pvalue:.4f}")

Variables standardisées (tau = 0.85) : corrigés 4, cassés 0, gain net +4, McNemar p = 0.1250


Le gain baisse et n'est plus significatif : il dépend de réglages qui semblent anodins.